# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://youthforseva.org/")
links

['#main',
 'https://youthforseva.org/',
 'https://youthforseva.org/',
 '/volunteer/',
 '/volunteer/sevacircle/',
 '/fundraising-campaigns/',
 '/corporate/',
 '/partnerngo/',
 '/careers/',
 'https://youthforseva.org/events/',
 'https://youthforseva.org/programs/',
 'https://youthforseva.org/education/',
 '/abhyasikas-learning-centres/',
 '/programs/chiguru/',
 '/programs/inspired-teachers-network/',
 '/programs/nmms/',
 '/programs/school-adoption-program/',
 '/programs/gift-a-school-kit/',
 '/programs/spoken-english-program/',
 '/programs/stem/',
 '/programs/swalpa-kali-swalpa-nali/',
 '/programs/swalpa-kali-swalpa-nali/',
 '/programs/vidya-sarathi/',
 '/programs/vidya-sethu/',
 'https://youthforseva.org/programs/vidyachetana',
 '/health/',
 '/programs/arogya-nidhi/',
 '/programs/basic-care-life-support/',
 '/programs/chaap/',
 '/programs/community-health/',
 '/programs/menstrual-hygiene/',
 '/programs/preventive-care/',
 'https://youthforseva.org/programs/school-health-program/',
 '/pr

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://youthforseva.org/"))


Here is the list of links on the website https://youthforseva.org/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#main
https://youthforseva.org/
https://youthforseva.org/
/volunteer/
/volunteer/sevacircle/
/fundraising-campaigns/
/corporate/
/partnerngo/
/careers/
https://youthforseva.org/events/
https://youthforseva.org/programs/
https://youthforseva.org/education/
/abhyasikas-learning-centres/
/programs/chiguru/
/programs/inspired-teachers-network/
/programs/nmms/
/programs/school-adoption-program/
/programs/gift-a-school-kit/
/programs/spoken-english-program/
/programs/stem/
/programs/swalpa-kali-swalpa-nali/
/programs/swalpa-kali-swalpa-nali/
/programs/vidya-sarathi/
/programs/vidya-sethu/
https://youthforseva.org/programs/vidyachetana
/health/
/programs/arogya-nidhi/
/programs/basic-care-life-supp

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://youthforseva.org/")

{'links': [{'type': 'about page', 'url': 'https://youthforseva.org/aboutus/'},
  {'type': 'vision and mission',
   'url': 'https://youthforseva.org/vision-and-mission/'},
  {'type': 'philosophy of seva',
   'url': 'https://youthforseva.org/our-philosophy-of-seva/'},
  {'type': 'trustees', 'url': 'https://youthforseva.org/trustees/'},
  {'type': 'financials', 'url': 'https://youthforseva.org/financials/'},
  {'type': 'reports', 'url': 'https://youthforseva.org/reports/'},
  {'type': 'careers page', 'url': 'https://youthforseva.org/careers/'},
  {'type': 'volunteer page', 'url': 'https://youthforseva.org/volunteer/'},
  {'type': 'partner NGO', 'url': 'https://youthforseva.org/partnerngo/'},
  {'type': 'corporate page', 'url': 'https://youthforseva.org/corporate/'},
  {'type': 'fundraising campaigns',
   'url': 'https://youthforseva.org/fundraising-campaigns/'},
  {'type': 'events', 'url': 'https://youthforseva.org/events/'},
  {'type': 'programs overview', 'url': 'https://youthforseva.or

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://youthforseva.org/")

Selecting relevant links for https://youthforseva.org/ by calling gpt-5-nano
Found 77 relevant links


{'links': [{'type': 'home page', 'url': 'https://youthforseva.org/'},
  {'type': 'about us', 'url': 'https://youthforseva.org/aboutus/'},
  {'type': 'vision and mission',
   'url': 'https://youthforseva.org/vision-and-mission/'},
  {'type': 'philosophy of seva',
   'url': 'https://youthforseva.org/our-philosophy-of-seva/'},
  {'type': 'trustees', 'url': 'https://youthforseva.org/trustees/'},
  {'type': 'financials', 'url': 'https://youthforseva.org/financials/'},
  {'type': 'reports', 'url': 'https://youthforseva.org/reports/'},
  {'type': 'contact us', 'url': 'https://youthforseva.org/contact-us/'},
  {'type': 'get involved', 'url': 'https://youthforseva.org/get-involved/'},
  {'type': 'careers', 'url': 'https://youthforseva.org/careers/'},
  {'type': 'volunteer', 'url': 'https://youthforseva.org/volunteer/'},
  {'type': 'volunteer circle',
   'url': 'https://youthforseva.org/volunteer/sevacircle/'},
  {'type': 'fundraising campaigns',
   'url': 'https://youthforseva.org/fundraising-c

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://youthforseva.org/"))

Selecting relevant links for https://youthforseva.org/ by calling gpt-5-nano
Found 27 relevant links
## Landing Page:

Volunteering in India for Education, Health & More | Youth for Seva

Skip to content
Home
Get Involved
Volunteering
SevaCircle
Ways to Give
CSR Collaborations
Partner NGOs
Careers
Events
Programs
Education
Abhyasikas Learning Centers
Chiguru
Inspired Teachers Network
NMMS
School Adoption Program
School Kit Drive
Spoken English Initiative
STEM
Summer Camp
Vidya Sarathi (Career Guidance)
Vidya Sethu (Bridge Course)
Vidyachetana
Health
Arogya Nidhi
Basic Care Life Support
Child Health and Awareness Program
Community Health
Menstrual Hygiene
Preventive Care
School Health Program
Telemedicine
Integrated Rural Development
Swagrama Fellowship
Livelihood
Women Empowerment
Reviving Our Rural Economy
Environment
Plantation
River/Lake Cleanup Drives
Strategic Partnerships
Government Partnerships
News & Stories
Newsletters
Media Coverage
Resources
Blogs
Our Impact
Social Impact
Sh

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("Youth for Seva", "https://youthforseva.org/")

Selecting relevant links for https://youthforseva.org/ by calling gpt-5-nano
Found 26 relevant links


'\nYou are looking at a company called: Youth for Seva\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nVolunteering in India for Education, Health & More | Youth for Seva\n\nSkip to content\nHome\nGet Involved\nVolunteering\nSevaCircle\nWays to Give\nCSR Collaborations\nPartner NGOs\nCareers\nEvents\nPrograms\nEducation\nAbhyasikas Learning Centers\nChiguru\nInspired Teachers Network\nNMMS\nSchool Adoption Program\nSchool Kit Drive\nSpoken English Initiative\nSTEM\nSummer Camp\nVidya Sarathi (Career Guidance)\nVidya Sethu (Bridge Course)\nVidyachetana\nHealth\nArogya Nidhi\nBasic Care Life Support\nChild Health and Awareness Program\nCommunity Health\nMenstrual Hygiene\nPreventive Care\nSchool Health Program\nTelemedicine\nIntegrated Rural Development\nSwagrama Fellowship\nLivelihood\nWomen Empowerment\nReviving Our Rural Economy\nEnvironment\nPl

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("Youth for Seva", "https://youthforseva.org/")

Selecting relevant links for https://youthforseva.org/ by calling gpt-5-nano
Found 22 relevant links


# Youth for Seva: Empowering India through Volunteering

---

## Who We Are

Youth for Seva is a dynamic volunteering organization dedicated to driving social impact in India across diverse sectors such as Education, Health, Integrated Rural Development, Livelihood, Women Empowerment, and Environment. With a vision to engage the youth in meaningful community service, we facilitate opportunities to contribute towards nation-building through impactful volunteer programs.

---

## Our Vision & Mission

- **Vision:** To create a vibrant volunteering culture in India contributing to holistic social development.
- **Mission:** To nurture and channelize youth energy into serving society effectively and sustainably.

Our philosophy centers around fostering long-term volunteer engagement and building self-sufficient communities through education, health care, livelihood support, environmental conservation, and social awareness programs.

---

## Key Programs & Initiatives

### Education
- **Abhyasikas Community Learning Centers:** Enhancing learning opportunities for underprivileged children.
- **Chiguru & Inspired Teachers Network:** Supporting teaching excellence in rural schools.
- **School Adoption Program & School Kit Drive:** Improving school infrastructure and resources.
- **Spoken English Initiative & STEM Education:** Empowering students with critical communication and technical skills.
- **Vidya Sarathi & Vidya Sethu:** Career guidance and bridge courses for student empowerment.

### Health
- **Arogya Nidhi Medical Fund:** Supporting medical needs of the poor.
- **Basic Care Life Support Training:** First responder skills dissemination.
- **Child Health and Awareness Program (CHAAP):** Adolescent health education.
- **Menstrual Hygiene & Preventive Care Awareness:** Promoting women's health.
- **Community Health & Telemedicine:** Extending healthcare access to rural areas.

### Integrated Rural Development
- **Swagrama Fellowship:** Engaging youth in rural upliftment and sustainable development programs.
- **Livelihood Programs & Women Empowerment:** Fostering economic independence.
- **Reviving Our Rural Economy:** Strengthening local economies with holistic development.

### Environment
- **Plantation Drives & River/Lake Cleanups:** Environmental conservation and community awareness.

---

## Our Impact

- Active chapters across major Indian cities including Ahmedabad, Delhi, Bengaluru, Hyderabad, and many more.
- Collaborative partnerships with government bodies, NGOs, and corporate CSR projects.
- Thousands of volunteers engaged in creating measurable and sustainable social change.
- Numerous success stories reflected through our newsletter, media coverage, and documented testimonials.

---

## Join Us

### Volunteering Opportunities
Youth for Seva is the ideal platform for anyone passionate about contributing to society. Whether you have a few hours or want to commit long-term, there are multiple roles to fit your skills and interests.

### Careers & Internships
Join our dedicated team that drives innovative social initiatives. We offer career opportunities that allow you to work at the crossroads of development, education, and community service.

### Ways to Give
Support our mission through donations, corporate CSR partnerships, or by volunteering your time and expertise via our SevaCircle platform.

---

## Company Culture

At Youth for Seva, we foster a culture of empathy, commitment, inclusivity, and leadership. We empower young volunteers and staff to be changemakers by providing mentorship, capacity building, and platforms for meaningful impact. Collaboration and transparency are core to how we engage with our partners and communities.

---

## Contact & Get Involved

- **Website:** youthforseva.org
- **Volunteer:** Engage in programs near you across India.
- **Donate:** Contribute to driving positive change.
- **Careers:** Explore opportunities to build a rewarding career in social development.

---

Youth for Seva invites you to be a part of India's journey to a better tomorrow by contributing your time, talent, and resources to the service of society. Together, let’s transform lives through the power of volunteering!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
stream_brochure("Youth for Seva", "https://youthforseva.org/")

Selecting relevant links for https://youthforseva.org/ by calling gpt-5-nano
Found 58 relevant links


# Youth for Seva: Empowering Communities Through Volunteerism

---

## About Us

Youth for Seva is a dynamic non-profit organization committed to fostering meaningful volunteerism across India, focusing on critical social sectors such as Education, Health, Integrated Rural Development, Livelihood, Women Empowerment, and Environment. With chapters in multiple cities including Ahmedabad, Delhi, Mysuru, Bengaluru, Hyderabad, and more, Youth for Seva creates impactful community programs that address real-world challenges by mobilizing passionate volunteers.

---

## Our Vision & Mission

We envision a society where youth actively contribute to sustainable social change through selfless service. Our mission is to provide a platform that enables volunteers to work alongside local communities and partner organizations, improving lives by enhancing education, health, and livelihood opportunities. We believe in empowering individuals to serve with dedication, compassion, and innovative thinking.

---

## Key Programs and Initiatives

### Education  
- **Abhyasikas Learning Centers** - Community-based learning hubs supporting rural education.  
- **Inspired Teachers Network** - Mobilizing educators to improve teaching quality.  
- **School Adoption Program** and **School Kit Drive** - Supporting under-resourced schools with resources and mentorship.  
- **Spoken English Initiative** & **STEM Education** - Enhancing skills for rural students.  
- **Career Guidance & Bridge Courses** with Vidya Sarathi and Vidya Sethu programs.

### Health  
- **Arogya Nidhi Medical Fund** - Assistance for critical healthcare needs.  
- **Basic Care Life Support Training** - Equipping volunteers in first responder skills.  
- **Child Health Awareness (CHAAP)** and **Menstrual Hygiene** programs for adolescent health education.  
- **Telemedicine Services** for rural and remote areas ensuring access to healthcare.

### Integrated Rural Development  
- **Swagrama Fellowship** empowering rural leadership and community projects.  
- **Livelihood and Women Empowerment Initiatives** - Sustainable economic development programs.  
- **Environmental Conservation** efforts including plantation drives and river/lake cleanups.

---

## Our Culture

Youth for Seva thrives on the values of empathy, collaboration, and innovation. We foster an inclusive environment where volunteers and employees are encouraged to take initiative and lead projects that make a tangible difference. Our multi-city chapters unite people from diverse backgrounds passionate about community service, creating a vibrant culture of learning and impact.

---

## Who We Serve

Our diverse customers and beneficiaries include children and youth from underprivileged rural and urban communities, women seeking empowerment through education and livelihood, patients in need of basic and preventive healthcare, and local populations striving for sustainable rural development. We also collaborate with corporate partners, government bodies, and NGOs to amplify social impact.

---

## Careers & Volunteer Opportunities

Youth for Seva welcomes individuals eager to contribute their skills and energy for a greater cause. Career opportunities and fellowships provide pathways for professionals and fresh graduates alike to engage deeply with social development. Volunteers can participate in a variety of programs tailored to their interests, gaining valuable experience while transforming lives.

**Career Paths:**
- Program Management  
- Community Engagement  
- Education & Training  
- Healthcare Support  
- Environmental Projects  

**Volunteer Roles:**
- Teaching and Mentoring  
- Health Awareness Campaigns  
- Event Coordination  
- Fundraising and CSR Collaboration  

Explore current openings and volunteer registration on our website.

---

## Get Involved

Be part of the change! Support Youth for Seva by volunteering your time, donating, or partnering with us through CSR initiatives. Together, we can create lasting social impact and enrich countless lives across India.

**Website:** https://www.youthforseva.org  
**Contact:** info@youthforseva.org  

---

Youth for Seva — Harnessing the power of youth to serve communities, build futures, and transform India.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Youth for Seva", "https://youthforseva.org/")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>